*   The concise implementation uses PyTorch layers, loss functions, optimizers, and data loaders while preserving the same training control flow.

In [1]:
import math
import random
import numpy as np
import torch

# 4.5.1 Defining the Model

## 1. Intuition

* A concise softmax-regression model is a linear layer that maps flattened inputs to class logits.

* The model returns logits. The loss function will handle softmax internally.

## 2. Why this exists

*   PyTorch layers register parameters automatically, so optimizers can find and update them.

## 3. Examples

*   Define a linear classifier for tiny 2 x 2 images and 3 classes.

In [2]:
net = torch.nn.Sequential( # Sequential means run each layer in order
    torch.nn.Flatten(),
    torch.nn.Linear(4, 3), # Creates a linear transformation with 4 input features and 3 output features, then internally performs X @ W.T + b
                           # We use W.T because W.shape is (3, 4) or (out_features, in_features), so use W.T to represent (4,3) and therefore the linear layer
)

X = torch.randn(2, 1, 2, 2)
logits = net(X)
logits.shape
# X: (2, 1, 2, 2)
# Flatten -> (2, 4)
# Linear: (2, 4) @ (4, 3) + bias -> (2, 3)

torch.Size([2, 3])

## 4. Step-by-step breakdown

* `torch.nn.Flatten()` converts each image into a feature vector.

* `torch.nn.Linear(4, 3)` maps 4 pixels to 3 class logits.

* `torch.nn.Sequential` runs the listed layers in order.

* The output shape is `(2, 3)`: 2 examples and 3 classes.

## 5. Connection to ML systems

*   This replaces the manual flattening, weight matrix, bias vector, and logits computation.

## 6. Common confusion points

- The model returns logits, not probabilities.
- `Sequential` runs layers in the order provided.
- [**The input size to `Linear` must match flattened feature count**].
- Registered parameters can be found with `net.parameters()`.

# 4.5.2 Softmax Revisited

## 1. Intuition

* Softmax is still conceptually part of classification, but PyTorch's `CrossEntropyLoss` combines log-softmax and negative log likelihood internally.

* Log-softmax means taking logarithms after softmax in a numerically stable way.

## 2. Why this exists

*   Combining these steps avoids numerical problems and reduces unnecessary computation.

## 3. Examples

*   Use `CrossEntropyLoss` directly on logits.

In [ ]:
labels = torch.tensor([0, 2]) # Sample 0 -> class 0, Sample 1 -> class 2
loss_fn = torch.nn.CrossEntropyLoss()
loss = loss_fn(logits, labels) # Performs softmax first to calculate the probabilities of each logit, then -log(correct class probability) based on comparison to labels.
loss

*   Softmax is still useful for inspection.

In [4]:
probs = torch.softmax(logits, dim=1)
probs.sum(dim=1)

tensor([1., 1.], grad_fn=<SumBackward1>)

## 4. Step-by-step breakdown

* `CrossEntropyLoss` receives raw logits and integer labels.

* It internally applies a stable log-softmax and computes the correct-class loss.

* `torch.softmax` can be used after the model when you want readable probabilities.

* The row sums confirm the inspected probabilities add to 1.

## 5. Connection to ML systems

*   In real PyTorch training, pass logits to `CrossEntropyLoss`.
*   Use softmax mainly for interpretation or prediction confidence.

## 6. Common confusion points

- Do not softmax before `CrossEntropyLoss` in the standard case.
- Labels should be class IDs, not one-hot rows.
- Softmax probabilities are useful for inspection.
- Logits can be any real numbers.

## Labels in `CrossEntropyLoss`

### 1. Class IDs (what `CrossEntropyLoss` expects) ✅

For a classification problem with 3 classes:

```
Class 0 = cat
Class 1 = dog
Class 2 = bird
```

If the correct answer is **dog**, the label is:

```python
label = torch.tensor(1)
```

For a batch of samples:

```python
labels = torch.tensor([0, 2, 1])
```

This means:

```
Sample 0 → class 0
Sample 1 → class 2
Sample 2 → class 1
```

This is the format expected by PyTorch:

```python
loss = torch.nn.CrossEntropyLoss()(logits, labels)
```

---

### 2. One-hot encoded labels (not the standard format for `CrossEntropyLoss`) ❌

The same "dog" label would be represented as:

```
[0, 1, 0]
```

because:

```
Class 0 → 0
Class 1 → 1  ← correct class
Class 2 → 0
```

A batch of one-hot labels would look like:

```python
labels = torch.tensor([
    [1, 0, 0],
    [0, 0, 1],
    [0, 1, 0]
])
```

This is called **one-hot encoding**.

---

## Why does `CrossEntropyLoss` use class IDs?

`CrossEntropyLoss` uses the label to select the probability of the correct class.

Example:

```python
logits = torch.tensor([
    [2.0, 1.0, 0.5],
    [0.2, 0.5, 3.0]
])

labels = torch.tensor([0, 2])
```

The labels tell the loss function:

```
Sample 0 → use class 0
Sample 1 → use class 2
```

Conceptually:

```python
correct_probabilities = [
    probs[0, 0],
    probs[1, 2]
]
```

Then cross-entropy calculates:

\[
-\log(\text{correct class probability})
\]

---

## Summary

`CrossEntropyLoss` expects:

```python
labels = torch.tensor([0, 2, 1])
```

where each number is the index of the correct class.

It does **not** normally expect:

```python
labels = torch.tensor([
    [1, 0, 0],
    [0, 0, 1],
    [0, 1, 0]
])
```

where labels are represented as one-hot vectors.

The model outputs **logits**, and `CrossEntropyLoss` uses the integer class IDs to determine which logit corresponds to the correct class.

# 4.5.3 Training

## 1. Intuition

*   Concise classification training follows: zero gradients, compute logits, compute cross-entropy loss, call backward, step optimizer.

## 2. Why this exists

*   The framework shortens the code, but the training order stays the same as the from-scratch version.

## 3. Examples

*   One compact training epoch on tiny synthetic image data.

```
    Dataset
      ↓
    DataLoader creates mini-batches
      ↓
    Forward pass (Xb → logits)
      ↓
    CrossEntropyLoss calculates error
      ↓
    backward() computes gradients
      ↓
    optimizer.step() updates weights
      ↓
    repeat
```



In [5]:
X = torch.randn(8, 1, 2, 2)
y = torch.tensor([0, 1, 2, 1, 0, 2, 1, 0])

loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X, y), batch_size=4)
net = torch.nn.Sequential(
    torch.nn.Flatten(), # shape of X becomes (batch_size, 1*2*2) or (batch_size, 4)
    torch.nn.Linear(4, 3) # Calculates (batch_size, 4) @ (4, 3) = shape of (batch_size, 3) or shape of (4,3)
)
loss_fn = torch.nn.CrossEntropyLoss() # # Takes logits and integer class IDs. Internally performs log-softmax + negative log likelihood.
opt = torch.optim.SGD(net.parameters(), lr=0.1)

for Xb, yb in loader:
  opt.zero_grad(); # Clears previously accumulated gradients in PyTorch
  loss_fn(net(Xb), yb).backward(); # Forward pass: Xb -> logits;
                                   # CrossEntropyLoss compares logits against yb (softmax to get possibilities from logits, then -log(correct probability));
                                   # Backward pass: calculates gradients using the chain rule
  opt.step() # Update params from calculated gradients from backward()
loss_fn(net(X), y)

tensor(1.0653, grad_fn=<NllLossBackward0>)

## 4. Step-by-step breakdown

* The DataLoader returns batches of images and labels.

* The model produces logits.

* `CrossEntropyLoss` computes scalar loss from logits and class IDs.

* `backward()` computes gradients.

* `opt.step()` updates parameters.

## 5. Connection to ML systems

*   This is the standard skeleton of many PyTorch classification loops.

## 6. Common confusion points

- `zero_grad()` must happen before the next backward pass.
- The model should output one logit per class.
> * The model outputs a score for each class that represents how strongly it leans toward that class, before converting those scores into probabilities (via `softmax` in `CrossEntropy`).
- `CrossEntropyLoss` combines softmax-like behavior with loss computation.
- Synthetic data only checks code structure, not real accuracy.

# 4.5.4 Summary

## 1. Intuition

*   The concise classifier uses `Flatten`, `Linear`, `CrossEntropyLoss`, `SGD`, and `DataLoader`.

## 2. Why this exists

*   These objects package the manual softmax-regression implementation into reusable PyTorch components.

## 3. Examples

*   Map concise objects to manual concepts.

In [ ]:
mapping = {
    "Flatten": "reshape images into vectors",
    "Linear": "compute class logits",
    "CrossEntropyLoss": "stable softmax plus loss",
    "SGD": "update parameters",
}

## 4. Step-by-step breakdown

*   Each concise object replaces a manual operation.

*   The execution order is still the training-loop order.

*   The mapping is the key to reading concise code without losing the underlying mechanics.

## 5. Connection to ML systems

*   Later neural classifiers replace only the model body.
*   The output logits and cross-entropy pattern remain common.

## 6. Common confusion points

- Concise code should still be explainable line by line.
- The model outputs logits.
- The loss function owns the stable softmax-loss combination.
- Optimizer objects update registered parameters.

# 4.5.5 Exercises

## 1. Intuition

*   These exercises practice modifying concise classifier dimensions and reading outputs.

## 2. Why this exists

*   Shape control is the main practical skill for concise classification code.

## 3. Examples

*   Exercise 1: define a classifier for 4 by 4 grayscale images and five classes.

In [6]:
net5 = torch.nn.Sequential(torch.nn.Flatten(), torch.nn.Linear(16, 5)) # (3, 1*4*4) or (3, 16) @ (16, 5) = shape of (3, 5)
X5 = torch.randn(3, 1, 4, 4)
logits5 = net5(X5)
logits5.shape

torch.Size([3, 5])

*   Exercise 2: compute predictions from logits.

In [9]:
preds = torch.argmax(logits5, dim=1)
preds.shape # (3,)

torch.Size([3])

## 4. Step-by-step breakdown

* Exercise 1 checks flattened input size and class count.

* Exercise 2 checks prediction shape.

* 3 examples should produce 3 predicted class IDs.

## 5. Connection to ML systems

*   These checks prevent common mistakes before training on larger image datasets.

## 6. Common confusion points

- Flattened size is channels * height * width.
- Output size is # of classes.
- Predictions are class IDs.
- Prediction shape should match label shape.